# ⚛️ Módulo 3: Mecánica Molecular y Campos de Fuerza
## Actividad 3.5: Análisis Conformacional

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_03_mecanica_molecular/05_analisis_conformacional.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender y aplicar búsqueda sistemática de conformaciones
- Usar métodos estocásticos: Monte Carlo conformacional
- Generar y priorizar ensembles conformacionales con RDKit
- Analizar distribuciones de energía y poblaciones de Boltzmann
- Calcular propiedades promedio ponderadas sobre el ensemble
- Comparar métodos de búsqueda conformacional

---

## 📚 Introducción

El **análisis conformacional** estudia las diferentes disposiciones tridimensionales de una molécula que se interconvierten por rotaciones de enlaces simples, sin romper ni formar enlaces covalentes.

### ¿Por qué es importante?

- **Diseño de fármacos**: el conformero bioactivo determina la afinidad de unión
- **Propiedades físicas**: punto de fusión, solubilidad, cristalinidad
- **Reactividad**: el conformero más reactivo puede ser una conformación menor
- **Docking molecular**: requiere explorar el espacio conformacional del ligando

### Métodos de Búsqueda Conformacional

| Método | Base | Ventaja | Limitación |
|--------|------|---------|------------|
| **Sistemático** | Rotación de todos los diedros | Completo | Exponencial en N diedros |
| **Monte Carlo** | Muestreo aleatorio + criterio Metropolis | Escalable | No garantiza completitud |
| **DG/ETKDG** | Geometría de distancias | Rápido, 3D desde SMILES | Puede perder algunos conformeros |
| **Dinámica Molecular** | Simulación en temperatura | Físico, muestrea barreras | Costoso |
| **Basin Hopping** | MC + optimización local | Eficiente para globales | Parámetros complejos |

### Distribución de Boltzmann

La población relativa de cada conformero a temperatura $T$:

$$p_i = \frac{e^{-\Delta E_i / k_B T}}{\sum_j e^{-\Delta E_j / k_B T}}$$

A 298 K, una diferencia de energía de ~1.4 kcal/mol reduce la población en ~10×.

In [ ]:
!pip install rdkit-pypi numpy scipy matplotlib 2>/dev/null || \
  pip install rdkit numpy scipy matplotlib
print('✓ Dependencias instaladas')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.Chem import rdDistGeom
    RDKIT_OK = True
    print('✓ RDKit disponible')
except ImportError:
    RDKIT_OK = False
    print('⚠️  RDKit no disponible')

# Constante de Boltzmann en kcal/mol/K
kB = 1.987e-3  # kcal/mol/K
T_STD = 298.15  # K
kBT = kB * T_STD
print(f'✓ kBT a {T_STD} K = {kBT:.4f} kcal/mol')

## 1. Distribución de Boltzmann y Poblaciones Conformacionales

In [ ]:
def poblacion_boltzmann(energias_rel, T=298.15):
    """
    Calcula la distribución de Boltzmann para un conjunto de conformeros.
    energias_rel: energías relativas en kcal/mol (mínimo = 0)
    """
    kBT = 1.987e-3 * T
    pesos = np.exp(-np.array(energias_rel) / kBT)
    return pesos / pesos.sum()

# Ejemplo: conformeros del butano
conformeros = {
    'Anti (180°)':       0.000,
    'Gauche+ (60°)':     0.900,
    'Gauche- (-60°)':    0.900,
    'Ecl. parcial (120°)': 3.600,
    'Ecl. total (0°)':   6.100,
}

nombres = list(conformeros.keys())
energias = list(conformeros.values())

temperaturas = [200, 298, 500, 1000]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Poblaciones a distintas temperaturas
ax = axes[0]
x = np.arange(len(nombres))
width = 0.2
colores_t = ['#1565C0', '#2196F3', '#FF9800', '#F44336']
for i, (T, c) in enumerate(zip(temperaturas, colores_t)):
    pops = poblacion_boltzmann(energias, T) * 100
    ax.bar(x + i*width, pops, width, label=f'T={T} K', color=c, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels([n.split('(')[0].strip() for n in nombres], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Población (%)', fontsize=12)
ax.set_title('Poblaciones de Boltzmann\ndel Butano a distintas T', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Variación con T para anti y gauche
ax = axes[1]
T_arr = np.linspace(100, 1200, 300)
pop_anti = [poblacion_boltzmann(energias, T)[0]*100 for T in T_arr]
pop_gauche = [poblacion_boltzmann(energias, T)[1]*100 for T in T_arr]
pop_rest = [100 - pa - pg for pa, pg in zip(pop_anti, pop_gauche)]

ax.plot(T_arr, pop_anti, 'b-', linewidth=2.5, label='Anti (180°)')
ax.plot(T_arr, [p*2 for p in [poblacion_boltzmann(energias, T)[1]*100 for T in T_arr]],
       'g-', linewidth=2.5, label='Gauche total (±60°)')
ax.axvline(298, color='gray', linestyle='--', alpha=0.7, label='T = 298 K')
ax.set_xlabel('Temperatura (K)', fontsize=12)
ax.set_ylabel('Población (%)', fontsize=12)
ax.set_title('Variación de Poblaciones con T\n(Butano)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Análisis de Poblaciones de Boltzmann', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

pops_298 = poblacion_boltzmann(energias) * 100
print('\nPoblaciones a 298 K:')
for nom, e, p in zip(nombres, energias, pops_298):
    print(f'  {nom:25s}: ΔE = {e:.2f} kcal/mol → {p:5.1f}%')

## 2. Búsqueda Conformacional con RDKit (ETKDG)

RDKit implementa el algoritmo **ETKDGv3** (Experimental Torsion Knowledge Distance Geometry) que genera conformaciones usando estadísticas de estructuras cristalinas.

In [ ]:
def generar_conformeros(smiles, nombre, n_confs=100, ff='MMFF94'):
    """
    Genera, optimiza y prioriza conformeros usando RDKit ETKDGv3 + MMFF94.
    Retorna (mol, energias_rel, rmsd_matrix)
    """
    if not RDKIT_OK:
        print('RDKit no disponible. Simulando resultados...')
        # Simular ensemble de 50 conformeros con distribución exponencial
        n_sim = min(n_confs, 50)
        E_sim = np.sort(np.random.exponential(1.5, n_sim))
        E_sim -= E_sim.min()
        return None, E_sim, None

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.numThreads = 0
    AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)

    n_generados = mol.GetNumConformers()
    if n_generados == 0:
        print(f'No se generaron conformeros para {nombre}')
        return mol, np.array([]), None

    # Optimizar con MMFF94
    resultados = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500, numThreads=0)
    energias = [r[1] for r in resultados if r[0] == 0]
    ids_ok = [i for i, r in enumerate(resultados) if r[0] == 0]

    E_arr = np.array(energias)
    E_rel = E_arr - E_arr.min()

    # Calcular RMSD entre conformeros (primeros 20 para no tardar)
    n_rmsd = min(20, len(ids_ok))
    rmsd_mat = np.zeros((n_rmsd, n_rmsd))
    for i in range(n_rmsd):
        for j in range(i+1, n_rmsd):
            r = AllChem.GetConformerRMS(mol, ids_ok[i], ids_ok[j])
            rmsd_mat[i, j] = rmsd_mat[j, i] = r

    print(f'\n✓ {nombre}: {n_generados} generados, {len(ids_ok)} optimizados')
    print(f'  Rango de energías: {E_rel.min():.2f} – {E_rel.max():.2f} kcal/mol')
    print(f'  Conformeros dentro de 3 kcal/mol: {(E_rel < 3.0).sum()}')
    return mol, E_rel, rmsd_mat

# Analizar propano y n-butano
mol_prop, E_prop, rmsd_prop = generar_conformeros('CCC', 'Propano', n_confs=50)
mol_but, E_but, rmsd_but = generar_conformeros('CCCC', 'n-Butano', n_confs=100)
mol_hex, E_hex, rmsd_hex = generar_conformeros('CCCCCC', 'n-Hexano', n_confs=200)

In [ ]:
def analizar_ensemble(E_rel, nombre, T=298.15):
    """Análisis estadístico completo de un ensemble conformacional."""
    if len(E_rel) == 0:
        return

    kBT = 1.987e-3 * T
    pesos = np.exp(-E_rel / kBT)
    pesos /= pesos.sum()

    E_prom = np.average(E_rel, weights=pesos)
    n_activos = (E_rel < 3.0).sum()
    E_umbral = [1.0, 2.0, 3.0, 5.0]

    print(f'\n📊 Ensemble: {nombre} ({len(E_rel)} conformeros)')
    print(f'  E mínima:   {E_rel.min():.3f} kcal/mol')
    print(f'  E máxima:   {E_rel.max():.3f} kcal/mol')
    print(f'  <E> Boltz.: {E_prom:.3f} kcal/mol')
    print(f'  Dentro de:')
    for eu in E_umbral:
        n = (E_rel <= eu).sum()
        print(f'    {eu:.0f} kcal/mol: {n:4d} confs ({100*n/len(E_rel):.1f}%)')
    print(f'  Población del global: {pesos[np.argmin(E_rel)]*100:.1f}%')

    return pesos

# Analizar cada ensemble
p_prop = analizar_ensemble(E_prop, 'Propano')
p_but  = analizar_ensemble(E_but, 'n-Butano')
p_hex  = analizar_ensemble(E_hex, 'n-Hexano')

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [
    (E_prop, p_prop, 'Propano', '#2196F3'),
    (E_but,  p_but,  'n-Butano', '#4CAF50'),
    (E_hex,  p_hex,  'n-Hexano', '#FF9800'),
]

for ax, (E, p, nom, col) in zip(axes, datasets):
    if E is None or len(E) == 0:
        continue
    # Histograma ponderado por Boltzmann
    ax.hist(E, bins=25, weights=p if p is not None else None,
           color=col, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='green', linestyle='--', alpha=0.7, label='Global mínimo')
    ax.axvline(3.0, color='red', linestyle=':', alpha=0.7, label='3 kcal/mol')
    ax.set_xlabel('Energía relativa (kcal/mol)', fontsize=11)
    ax.set_ylabel('Fracción poblacional', fontsize=11)
    ax.set_title(f'{nom}\n({len(E)} conformeros)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribución Conformacional (RDKit ETKDGv3 + MMFF94)', 
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Monte Carlo Conformacional

El método de **Monte Carlo conformacional (MCMC)** perturba aleatoriamente los ángulos diedros y acepta/rechaza el movimiento con el criterio de Metropolis:

$$P_{\text{aceptar}} = \min\left(1,\; e^{-\Delta E / k_B T}\right)$$

In [ ]:
def mc_conformacional_1d(n_pasos=5000, T=298.15, delta_max=30.0,
                          E_func=None, seed=42):
    """
    Monte Carlo conformacional en 1D (un diedro).
    Usa la PES del butano como función de energía.
    """
    np.random.seed(seed)
    kBT = 1.987e-3 * T

    if E_func is None:
        def E_func(phi):
            p = np.radians(phi)
            V = (1.40*(1+np.cos(p)) + 0.54*(1-np.cos(2*p)) +
                 0.20*(1+np.cos(3*p)))
            return V

    phi_actual = np.random.uniform(-180, 180)
    E_actual = E_func(phi_actual)

    trayectoria = [phi_actual]
    energias_traj = [E_actual]
    n_aceptados = 0

    for paso in range(n_pasos):
        delta = np.random.uniform(-delta_max, delta_max)
        phi_nuevo = ((phi_actual + delta) + 180) % 360 - 180
        E_nuevo = E_func(phi_nuevo)
        delta_E = E_nuevo - E_actual

        if delta_E < 0 or np.random.random() < np.exp(-delta_E / kBT):
            phi_actual = phi_nuevo
            E_actual = E_nuevo
            n_aceptados += 1

        trayectoria.append(phi_actual)
        energias_traj.append(E_actual)

    tasa = n_aceptados / n_pasos * 100
    return np.array(trayectoria), np.array(energias_traj), tasa

# Comparar MC a diferentes temperaturas
temperaturas_mc = [100, 298, 600]
colores_mc = ['#1565C0', '#4CAF50', '#F44336']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
phi_ref = np.linspace(-180, 180, 360)
p = np.radians(phi_ref)
E_ref = 1.40*(1+np.cos(p)) + 0.54*(1-np.cos(2*p)) + 0.20*(1+np.cos(3*p))
E_ref -= E_ref.min()

for i, (T_mc, col) in enumerate(zip(temperaturas_mc, colores_mc)):
    traj, E_traj, tasa = mc_conformacional_1d(n_pasos=5000, T=T_mc)

    # Trayectoria MC
    ax = axes[0, i]
    ax.plot(range(len(traj)), traj, color=col, alpha=0.6, linewidth=0.5)
    ax.set_xlabel('Paso MC', fontsize=10)
    ax.set_ylabel('φ (°)', fontsize=10)
    ax.set_title(f'Trayectoria MC\nT={T_mc} K (tasa={tasa:.0f}%)', 
                fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Histograma vs PES
    ax = axes[1, i]
    ax_twin = ax.twinx()
    ax_twin.plot(phi_ref, E_ref, 'k-', linewidth=1.5, alpha=0.5, label='PES')
    ax.hist(traj, bins=60, color=col, alpha=0.7, density=True, label='MC')
    ax.set_xlabel('φ (°)', fontsize=10)
    ax.set_ylabel('Densidad', fontsize=10, color=col)
    ax_twin.set_ylabel('E (kcal/mol)', fontsize=10)
    ax.set_title(f'Histograma vs PES\nT={T_mc} K', fontsize=10, fontweight='bold')
    ax.set_xlim(-180, 180)
    ax.grid(True, alpha=0.3)

plt.suptitle('Monte Carlo Conformacional — Butano (diedro C−C−C−C)',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Búsqueda Sistemática y Filtrado de Duplicados

In [ ]:
def busqueda_sistematica_2d(E_func, n_puntos=24, umbral_E=5.0):
    """
    Búsqueda sistemática en 2D (dos diedros).
    E_func(phi1, phi2) debe retornar la energía.
    """
    angulos = np.linspace(-180, 165, n_puntos)
    resultados = []

    for phi1 in angulos:
        for phi2 in angulos:
            E = E_func(phi1, phi2)
            resultados.append({'phi1': phi1, 'phi2': phi2, 'E': E})

    import pandas as pd
    df = pd.DataFrame(resultados)
    df['E_rel'] = df['E'] - df['E'].min()

    # Filtrar por energía
    df_activos = df[df['E_rel'] <= umbral_E].copy()

    # Ordenar por energía
    df_activos = df_activos.sort_values('E_rel').reset_index(drop=True)

    print(f'\n📊 BÚSQUEDA SISTEMÁTICA 2D ({n_puntos}×{n_puntos} = {n_puntos**2} puntos)')
    print(f'   Puntos totales evaluados: {len(df)}')
    print(f'   Dentro de {umbral_E} kcal/mol: {len(df_activos)}')
    print(f'\n   Top 10 estructuras:')
    print(df_activos[['phi1','phi2','E_rel']].head(10).to_string(index=False))

    return df_activos

def pes_2d_simple(phi1, phi2):
    """PES 2D analítica de 1,2-dibromoetano."""
    p1, p2 = np.radians(phi1), np.radians(phi2)
    V1 = 1.5*(1+np.cos(p1)) + 0.6*(1-np.cos(2*p1)) + 0.3*(1+np.cos(3*p1))
    V2 = 1.5*(1+np.cos(p2)) + 0.6*(1-np.cos(2*p2)) + 0.3*(1+np.cos(3*p2))
    V12 = 0.2 * np.cos(p1 - p2)  # acoplamiento
    return V1 + V2 + V12

df_sys = busqueda_sistematica_2d(pes_2d_simple, n_puntos=24, umbral_E=4.0)

# Visualización
phi_arr = np.linspace(-180, 180, 100)
PHI1, PHI2 = np.meshgrid(phi_arr, phi_arr)
Z_2d = np.vectorize(pes_2d_simple)(PHI1, PHI2)
Z_2d -= Z_2d.min()

fig, ax = plt.subplots(figsize=(8, 6))
cp = ax.contourf(PHI1, PHI2, Z_2d, levels=20, cmap='RdYlGn_r')
ax.contour(PHI1, PHI2, Z_2d, levels=[1, 2, 3, 4], colors='white',
          alpha=0.4, linewidths=0.8)
plt.colorbar(cp, ax=ax, label='Energía relativa (kcal/mol)')
ax.scatter(df_sys['phi1'], df_sys['phi2'], c=df_sys['E_rel'],
          cmap='cool', s=60, zorder=5, edgecolors='white', linewidth=0.5,
          label='Puntos sistemáticos')
ax.set_xlabel('φ₁ (°)', fontsize=12)
ax.set_ylabel('φ₂ (°)', fontsize=12)
ax.set_title('Búsqueda Sistemática sobre PES 2D\n(1,2-dibromoetano modelo)',
            fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Propiedades Promedio del Ensemble

In [ ]:
def propiedades_promedio_ensemble(smiles, nombre, T=298.15):
    """
    Calcula propiedades moleculares promediadas sobre el ensemble conformacional.
    """
    if not RDKIT_OK:
        print('RDKit no disponible. Simulando...')
        return

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMultipleConfs(mol, numConfs=100, params=params)

    if mol.GetNumConformers() == 0:
        print(f'No se generaron conformeros para {nombre}')
        return

    resultados = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500)
    kBT = 1.987e-3 * T

    energias, radios_giro = [], []
    for i, r in enumerate(resultados):
        if r[0] != 0:
            continue
        energias.append(r[1])
        # Radio de giro
        conf = mol.GetConformer(i)
        pos = conf.GetPositions()
        centro = pos.mean(axis=0)
        Rg = np.sqrt(((pos - centro)**2).sum(axis=1).mean())
        radios_giro.append(Rg)

    if len(energias) == 0:
        return

    E_arr = np.array(energias)
    E_rel = E_arr - E_arr.min()
    pesos = np.exp(-E_rel / kBT)
    pesos /= pesos.sum()
    Rg_arr = np.array(radios_giro)

    print(f'\n📊 PROPIEDADES DEL ENSEMBLE — {nombre}')
    print(f'  Conformeros analizados: {len(energias)}')
    print(f'  Radio de giro:')
    print(f'    Mínimo global: {Rg_arr[np.argmin(E_rel)]:.3f} Å')
    print(f'    Promedio simple: {Rg_arr.mean():.3f} Å')
    print(f'    Promedio Boltzmann: {np.average(Rg_arr, weights=pesos):.3f} Å')

    # Gráfico
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Rg vs E
    sc = axes[0].scatter(E_rel, Rg_arr, c=pesos, cmap='YlOrRd', s=30, alpha=0.8)
    plt.colorbar(sc, ax=axes[0], label='Peso Boltzmann')
    axes[0].set_xlabel('Energía relativa (kcal/mol)', fontsize=11)
    axes[0].set_ylabel('Radio de giro (Å)', fontsize=11)
    axes[0].set_title(f'Rg vs E — {nombre}', fontsize=11, fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    # Histograma de Rg ponderado
    axes[1].hist(Rg_arr, bins=20, weights=pesos, color='steelblue',
                alpha=0.8, edgecolor='white')
    axes[1].axvline(np.average(Rg_arr, weights=pesos), color='red',
                   linestyle='--', linewidth=2, label='<Rg> Boltzmann')
    axes[1].set_xlabel('Radio de giro (Å)', fontsize=11)
    axes[1].set_ylabel('Fracción poblacional', fontsize=11)
    axes[1].set_title(f'Distribución de Rg — {nombre}', fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

propiedades_promedio_ensemble('CCCCCC', 'n-Hexano')
propiedades_promedio_ensemble('CCCCCCC', 'n-Heptano')

## 6. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Usa `generar_conformeros` con la aspirina (`CC(=O)Oc1ccccc1C(=O)O`):
1. Genera 200 conformeros y optimiza con MMFF94
2. ¿Cuántos están dentro de 2 kcal/mol del mínimo global?
3. Calcula la distribución de Boltzmann a 298 K

### Ejercicio 2 (Intermedio)
Compara los ensembles conformacionales de **n-pentano** y **neopentano** (`CC(C)(C)C`):
1. ¿Cuál tiene más conformeros únicos?
2. ¿Cuál tiene mayor radio de giro promedio?
3. ¿Cómo afecta la ramificación a la distribución conformacional?

### Ejercicio 3 (Avanzado)
Modifica `mc_conformacional_1d` para trabajar en 2D (dos diedros). Aplícalo al etano-1,2-diol (`OCC O`) y compara el histograma 2D con el mapa de contornos de la PES. ¿El MC converge correctamente a 298 K?

In [ ]:
# Ejercicio 1: Aspirina
mol_asp, E_asp, _ = generar_conformeros('CC(=O)Oc1ccccc1C(=O)O', 'Aspirina', n_confs=200)
if E_asp is not None and len(E_asp) > 0:
    p_asp = analizar_ensemble(E_asp, 'Aspirina')
# Tu código para ejercicios 2 y 3 aquí...

## 7. Referencias

1. Leach, A. R. (2001). *Molecular Modelling: Principles and Applications*, 2nd ed. Pearson.
2. Riniker, S. & Landrum, G. A. (2015). Better Informed Distance Geometry: Using What We Know To Improve Conformation Generation. *J. Chem. Inf. Model.*, 55(12), 2562–2574. (ETKDGv1)
3. Wang, S. et al. (2020). Improving Conformer Generation for Small Rings and Macrocycles Based on Distance Geometry and Experimental Torsional-Angle Preferences. *J. Chem. Inf. Model.*, 60(4), 2044–2058. (ETKDGv3)
4. Metropolis, N. et al. (1953). Equation of State Calculations by Fast Computing Machines. *J. Chem. Phys.*, 21, 1087.
5. Chang, G. et al. (1989). An internal coordinate Monte Carlo method for searching conformational space. *J. Am. Chem. Soc.*, 111(12), 4379–4386.

---

## 📚 Recursos Adicionales

### Herramientas
- [RDKit ETKDGv3](https://www.rdkit.org/docs/GettingStartedInPython.html#generating-3d-coordinates) — Generación de conformeros
- [CONFAB](https://open-babel.readthedocs.io/) — Búsqueda sistemática con OpenBabel
- [OMEGA (OpenEye)](https://www.eyesopen.com/omega) — Herramienta comercial de referencia
- [RMS Diversity Picker](https://www.rdkit.org/) — Selección de conformeros diversos

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Explicar la distribución de Boltzmann y calcular poblaciones conformacionales
- ✅ Generar ensembles conformacionales con RDKit ETKDGv3 y MMFF94
- ✅ Implementar y analizar una búsqueda Monte Carlo conformacional
- ✅ Realizar búsquedas sistemáticas en 2D y visualizar resultados
- ✅ Calcular propiedades promedio ponderadas sobre el ensemble
- ✅ Comparar la eficiencia de búsqueda sistemática vs MC

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 3.5: Análisis Conformacional**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_3.4-Superficies_PES-blue.svg)](04_superficies_energia.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_3.6_➡️-Cálculo_Propiedades-green.svg)](06_calculo_propiedades.ipynb)

---

📚 **[Volver al Módulo 3](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G - 2026*

</div>